# MOIRAI 2 compatibility probe
Run in a separate fresh Colab Pro+ runtime. This notebook validates official inference only. It intentionally does not invent a MOIRAI 2 full-fine-tuning loss or reuse a MOIRAI 1.x recipe.

In [ ]:
%pip install -q -r requirements/moirai.txt
!git rev-parse HEAD
!python -m pip freeze

In [ ]:
import hashlib, json, os, platform, time
from pathlib import Path
import pandas as pd
import torch
from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module
os.environ.setdefault('HF_HOME', '.cache/moirai')
REPO = 'Salesforce/moirai-2.0-R-small'; REVISION = '30f43ff08c8494f4943ae1521e9d4e94a0fbb389'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print({'python': platform.python_version(), 'torch': torch.__version__, 'device': str(DEVICE)})

In [ ]:
import subprocess, sys
base=[sys.executable,'scripts/probe_moirai_cpu.py','--data','data/official_raw/ett/ETTh1.csv','--cache-dir','.cache/moirai']
multi=subprocess.run([*base,'--channels','7'],capture_output=True,text=True)
assert multi.returncode != 0 and 'Shape mismatch' in multi.stderr
failure_summary='official 7-channel multi-token forward shape mismatch'
single=subprocess.run([*base,'--channels','1'],capture_output=True,text=True,check=True)
univariate_runtime=json.loads(single.stdout)
print({'multivariate_status':'inference_failed','univariate_status':'cpu_or_gpu_validated','univariate_shapes':[univariate_runtime['synthetic_quantile_shape'],univariate_runtime['synthetic_point_shape']]})

In [ ]:
runtime={'model':REPO,'revision':REVISION,'device':str(DEVICE),'multivariate_status':'inference_failed','multivariate_failure':failure_summary,'univariate_probe':univariate_runtime,'finetune_steps':0,'finetune_status':'unsupported_by_pinned_official_API','test_split_used':False}
Path('results/manifests/models').mkdir(parents=True,exist_ok=True); Path('results/manifests/models/moirai_runtime_probe.json').write_text(json.dumps(runtime,indent=2),encoding='utf-8')
runtime